# Exercise: Feed Clean Documents into Your RAG Pipeline

In the walkthrough you built the RAG pipeline by hand on text pulled straight from a PDF reader. Real documents are messier than that, and the quality of the text you feed in directly shapes the quality of what you retrieve.

In this exercise you will reuse the previous pipeline but we convert the document to Markdown with
[MarkItDown](https://github.com/microsoft/markitdown), clean it, and then **measure** how much that cleaning changes what the pipeline retrieves and answers.

> **Our run (2026-11-27): a local model.** This notebook was run with a free model on this
> computer instead of the Groq cloud model the text names: `qwen3.5:4b`, served by
> [Ollama](https://ollama.com) (4.2 billion parameters, a 3.3 GB file). The embedding model
> `all-mpnet-base-v2` is local too. No API key was used and no text left the computer.
> Where the lesson says "Groq" or "gpt-oss", read "the local model". A small model is
> less precise than a large cloud model, so check every answer against the retrieved passages.

In [1]:
# Our run: pick the local Ollama model for this notebook process only.
# The .env file is not touched: variables set here win over it.
import os

os.environ["LLM_PROVIDER"] = "ollama"
os.environ["OLLAMA_MODEL"] = "qwen3.5:4b"

## Learning Objectives

By completing this exercise, you will be able to:

- Convert a PDF to Markdown with MarkItDown and inspect the raw conversion.
- Write cleaning rules that remove common extraction artifacts.
- Measure the effect of cleaning on retrieval with similarity scores and chunk overlap.

## Setup

Load your credentials. The shared chat-model adapter reads the selected provider and matching API key from `.env`.

In [2]:
from dotenv import load_dotenv

load_dotenv()

PDF = "documents/react_paper.pdf"
QUERY = "What is the ReAct framework and how does it combine reasoning and acting?"

## Provided: the RAG pipeline from the walkthrough

Run this cell as is. It gives you two things:

- `run_rag(text, query)`, which chunks a body of text, embeds it, retrieves the most
  relevant chunks, and generates an answer. It returns the `answer`, the retrieved
  `chunks`, and their cosine similarity `scores`.
- `retrieval_overlap(chunks_a, chunks_b)`, the Jaccard overlap between two sets of
  retrieved chunks (0 means completely different, 1 means identical).

You do not need to change anything here.

In [3]:
import numpy as np
from sentence_transformers import SentenceTransformer

from llm_config import create_chat_model, response_text

model = create_chat_model()
embedder = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")


def chunk_text(text, size=800, overlap=100):
    if size <= 0 or not 0 <= overlap < size:
        raise ValueError("Require size > 0 and 0 <= overlap < size.")
    chunks, start = [], 0
    while start < len(text):
        chunks.append(text[start : start + size])
        start += size - overlap
    return [c.strip() for c in chunks if c.strip()]


def retrieve(query, chunks, chunk_vectors, k=3):
    query_vector = embedder.encode([query], normalize_embeddings=True)[0]
    scores = chunk_vectors @ query_vector
    top = np.argsort(scores)[::-1][:k]
    return [(chunks[i], float(scores[i])) for i in top]


def generate_answer(query, context):
    prompt = (
        "Answer the question using only the context below.\n\n"
        "Context:\n" + "\n---\n".join(context) + f"\n\nQuestion: {query}\nAnswer:"
    )
    response = model.invoke([{"role": "user", "content": prompt}])
    return response_text(response)


def run_rag(text, query, k=3):
    """Chunk, embed, retrieve, and generate. Returns answer, chunks, and scores."""
    chunks = chunk_text(text)
    chunk_vectors = embedder.encode(chunks, normalize_embeddings=True)
    hits = retrieve(query, chunks, chunk_vectors, k=k)
    context = [chunk for chunk, _ in hits]
    return {
        "answer": generate_answer(query, context),
        "chunks": context,
        "scores": [score for _, score in hits],
    }


def retrieval_overlap(chunks_a, chunks_b):
    """Jaccard overlap between two lists of retrieved chunks (0 to 1)."""
    a, b = set(chunks_a), set(chunks_b)
    return len(a & b) / len(a | b) if (a or b) else 0.0

## Exercise 1: Convert the document to Markdown

Use MarkItDown to convert the PDF to a Markdown string.

Hints:

- The API is `MarkItDown().convert(path).text_content`.
- After converting, print the first 1000 characters and look closely. Real PDF
  extractions often have missing spaces, repeated page headers, and stray line breaks.
  Note the problems you see: you will fix them next.

In [4]:
from markitdown import MarkItDown


def to_markdown(path):
    """Convert a document to a Markdown string with MarkItDown."""
    # MarkItDown reads the file and returns a result object.
    # The Markdown text sits in its .text_content field.
    return MarkItDown().convert(path).text_content

In [5]:
raw_text = to_markdown(PDF)
print(f"{len(raw_text):,} characters in {raw_text.count(chr(10)) + 1:,} lines\n")
print(raw_text[:1000])

205,117 characters in 2,017 lines

PublishedasaconferencepaperatICLR2023
REACT: SYNERGIZING REASONING AND ACTING IN
LANGUAGE MODELS
ShunyuYao∗*,1,JeffreyZhao2,DianYu2,NanDu2,IzhakShafran2,KarthikNarasimhan1,YuanCao2
1DepartmentofComputerScience,PrincetonUniversity
2GoogleResearch,Brainteam
1{shunyuy,karthikn}@princeton.edu
2{jeffreyzhao,dianyu,dunan,izhak,yuancao}@google.com
ABSTRACT
Whilelargelanguagemodels(LLMs)havedemonstratedimpressiveperformance
across tasks in language understanding and interactive decision making, their
abilitiesforreasoning(e.g. chain-of-thoughtprompting)andacting(e.g. action
plangeneration)haveprimarilybeenstudiedasseparatetopics. Inthispaper,we
exploretheuseofLLMstogeneratebothreasoningtracesandtask-specificactions
inaninterleavedmanner,allowingforgreatersynergybetweenthetwo: reasoning
traces help the model induce, track, and update action plans as well as handle
exceptions,whileactionsallowittointerfacewithandgatheradditionalinformation
fromexternalsourcessu

Reading 1000 characters shows the problems. Counting them over the whole text shows how big each one is (my own extra cell).

In [6]:
import re
from collections import Counter

lines = raw_text.split("\n")

# the same line on every page is a page header
top_line, repeats = Counter(l.strip() for l in lines if l.strip()).most_common(1)[0]
print("most repeated line:", repr(top_line), "\n")

problems = {
    "times the repeated header appears": repeats,
    "glyph codes like (cid:71)": len(re.findall(r"\(cid:\d+\)", raw_text)),
    "table lines (they start with |)": sum(l.strip().startswith("|") for l in lines),
    "characters inside those table lines": sum(
        len(l) for l in lines if l.strip().startswith("|")
    ),
    "lines that are only a number": sum(
        bool(re.fullmatch(r"\s*\d{1,3}\s*", l)) for l in lines
    ),
    "lines of 1 to 3 characters": sum(0 < len(l.strip()) <= 3 for l in lines),
    "lines that end with a hyphen": sum(l.rstrip().endswith("-") for l in lines),
    "long lines with almost no spaces": sum(
        len(l) > 40 and l.count(" ") / len(l) < 0.05 for l in lines
    ),
}
for name, n in problems.items():
    print(f"{n:>8,}  {name}")

most repeated line: 'PublishedasaconferencepaperatICLR2023' 

      33  times the repeated header appears
   7,127  glyph codes like (cid:71)
     440  table lines (they start with |)
  84,926  characters inside those table lines
      41  lines that are only a number
     102  lines of 1 to 3 characters
      26  lines that end with a hyphen
     625  long lines with almost no spaces


**What I see in the raw conversion**

- **Glued words.** Most lines have lost their spaces: `Whilelargelanguagemodels(LLMs)havedemonstrated...`. 625 long lines have almost no spaces.
- **A repeated page header.** `PublishedasaconferencepaperatICLR2023` appears 33 times, once per page.
- **Glyph codes.** 7,127 codes like `(cid:71)`. They are the text of figures whose font the converter could not read.
- **Tables full of padding.** 440 lines start with `|`. Together they hold 84,926 characters, mostly spaces and dashes.
- **Page furniture.** 41 lines are only a page number, and 102 lines have 1 to 3 characters.
- **Words split by a hyphen** at a line end: 26 lines.

The raw text has 205,117 characters. The same paper read with `pypdf` in the lesson had 110,116. So almost half of this raw text is not content.

## Exercise 2: Clean the Markdown

Write a cleaning function that fixes the problems you spotted. Decide which rules
matter for this document.

Ideas to consider:

- Collapse runs of blank lines and strip trailing whitespace.
- Drop lines that are just a page number or the repeated conference header.
- Remove very short lines that are page furniture rather than content.
- Rejoin words split across a line break by a hyphen.

Store the result as `clean_text` and print the character count before and after so you
can see how much you removed.

In [7]:
import re

# the page header, exactly as MarkItDown returns it (the spaces are missing)
HEADER = "PublishedasaconferencepaperatICLR2023"

# Each rule is one small function: text in, cleaner text out.
# That way every rule can also be tested alone (see Exercise 3).


def drop_glyph_codes(text):
    """(cid:71) codes are letters of a figure the converter could not read."""
    return re.sub(r"\(cid:\d+\)", "", text)


def unwrap_tables(text):
    """Drop the dashed table rows, the | bars and the padding spaces."""
    kept = []
    for line in text.split("\n"):
        if re.fullmatch(r"\s*\|[\s\-|]*\|\s*", line):
            continue  # a row of dashes or of empty cells
        if line.strip().startswith("|"):
            line = line.replace("|", " ")
        kept.append(re.sub(r"[ \t]+", " ", line).strip())
    return "\n".join(kept)


def drop_header_and_page_numbers(text):
    """Drop the repeated conference header and lines that are only a number."""
    kept = [
        line
        for line in text.split("\n")
        if line.strip() != HEADER and not re.fullmatch(r"\s*\d{1,3}\s*", line)
    ]
    return "\n".join(kept)


def drop_short_lines(text):
    """Lines of 1 to 3 characters are page furniture (axis labels, 'OK.')."""
    return "\n".join(l for l in text.split("\n") if not 0 < len(l.strip()) <= 3)


def join_hyphenated_words(text):
    """'Con-' + line break + 'sider' becomes 'Consider'."""
    return re.sub(r"(\w)-\n(\w)", r"\1\2", text)


def collapse_blank_lines(text):
    """Strip trailing spaces and squeeze runs of blank lines into one."""
    text = "\n".join(line.rstrip() for line in text.split("\n"))
    return re.sub(r"\n{3,}", "\n\n", text).strip()


RULES = [
    drop_glyph_codes,
    unwrap_tables,
    drop_header_and_page_numbers,
    drop_short_lines,
    join_hyphenated_words,
    collapse_blank_lines,
]


def clean_markdown(text):
    """Return a cleaned version of the Markdown text."""
    for rule in RULES:
        text = rule(text)
    return text

In [8]:
clean_text = clean_markdown(raw_text)

removed = len(raw_text) - len(clean_text)
print(f"before:  {len(raw_text):>8,} characters")
print(f"after:   {len(clean_text):>8,} characters")
print(f"removed: {removed:>8,} characters ({removed / len(raw_text):.0%})\n")
print(clean_text[:600])

before:   205,117 characters
after:     94,370 characters
removed:  110,747 characters (54%)

REACT: SYNERGIZING REASONING AND ACTING IN
LANGUAGE MODELS
ShunyuYao∗*,1,JeffreyZhao2,DianYu2,NanDu2,IzhakShafran2,KarthikNarasimhan1,YuanCao2
1DepartmentofComputerScience,PrincetonUniversity
2GoogleResearch,Brainteam
1{shunyuy,karthikn}@princeton.edu
2{jeffreyzhao,dianyu,dunan,izhak,yuancao}@google.com
ABSTRACT
Whilelargelanguagemodels(LLMs)havedemonstratedimpressiveperformance
across tasks in language understanding and interactive decision making, their
abilitiesforreasoning(e.g. chain-of-thoughtprompting)andacting(e.g. action
plangeneration)haveprimarilybeenstudiedasseparatetopics. Inthispa


The six rules remove 110,747 of the 205,117 characters (54 %).

**What the rules cannot repair:** the missing spaces. The words are still glued (`Whilelargelanguagemodels`). A search pattern cannot know where one word ends and the next begins. That would need a word-splitting tool with a dictionary, or a converter that keeps the spaces. I left the glued words as they are and measure the effect below.

## Exercise 3: Measure the effect of cleaning

This is the payoff. Run the provided pipeline twice on the same question, once with
`raw_text` and once with `clean_text`, then compare them with two metrics.

**Metric 1: mean retrieval score.** Average the `scores` of the retrieved chunks for
each run. A higher mean suggests the retrieved chunks match the query more closely.

**Metric 2: retrieval overlap.** Use `retrieval_overlap(...)` on the two `chunks`
lists. A low overlap means cleaning changed *which* chunks were retrieved.

> Read the scores as a signal, not a verdict. The raw and cleaned runs have different
> chunk sets, so the numbers are not a perfect like-for-like comparison. Always read the
> retrieved chunks and the answer alongside the metrics.

Tasks:

- Call `run_rag` on `raw_text` and on `clean_text`.
- Print the mean retrieval score for each run and the overlap between them.
- Print both answers, then write a sentence on what changed and why.

In [9]:
raw_result = run_rag(raw_text, QUERY)
clean_result = run_rag(clean_text, QUERY)


def mean(values):
    return sum(values) / len(values)


print("raw scores:        ", [round(s, 3) for s in raw_result["scores"]])
print("cleaned scores:    ", [round(s, 3) for s in clean_result["scores"]])
print("raw mean score:    ", round(mean(raw_result["scores"]), 3))
print("cleaned mean score:", round(mean(clean_result["scores"]), 3))
print(
    "retrieval overlap: ",
    retrieval_overlap(raw_result["chunks"], clean_result["chunks"]),
)
print("\nraw answer:\n", raw_result["answer"])
print("\ncleaned answer:\n", clean_result["answer"])

raw scores:         [0.556, 0.488, 0.487]
cleaned scores:     [0.575, 0.568, 0.517]
raw mean score:     0.51
cleaned mean score: 0.553
retrieval overlap:  0.0

raw answer:
 Based on the provided context, **ReAct** is a framework that integrates decision-making and reasoning capabilities directly into a large language model. It combines reasoning and acting by allowing the model to decide the asynchronous occurrence of thoughts and actions for itself through a flexible and sparse thought-action format. Specifically, this is achieved by having human annotators type down their thoughts in language on top of their actions taken, creating prompts that do not require ad-hoc choices or specific design constraints. This structure enables the model to perform diverse reasoning types (such as goal decomposition and commonsense reasoning) before executing actions, making its problem-solving trajectory more grounded, fact-driven, and trustworthy compared to methods like IM-style prompting.

cleane

Always read the chunks next to the numbers. Here is the best chunk of each run (first 300 characters).

In [10]:
print("best raw chunk:\n", raw_result["chunks"][0][:300])
print("\nbest cleaned chunk:\n", clean_result["chunks"][0][:300])

best raw chunk:
 mostrelevantpositionsofatrajectory,soweletthelanguagemodeldecidethe
asynchronousoccurrenceofthoughtsandactionsforitself.
Sincedecisionmakingandreasoningcapabilitiesareintegratedintoalargelanguagemodel,ReAct
enjoys several unique features: A) Intuitive and easy to design: Designing ReAct prompts is
s

best cleaned chunk:
 ationsteps. Incontrast,fordecision
makingtasksthatpotentiallyinvolvealargenumberofactions(Figure1(2)),thoughtsonlyneedto
1WeshowsomeGPT-3(Brownetal.,2020)resultsinAppendixA.1,whichoutperformsPaLM-540B.

appearsparselyinthemostrelevantpositionsofatrajectory,soweletthelanguagemodeldecidethe
asynchrono


**What changed and why.** Cleaning raised the mean score of the three retrieved chunks from 0.510 to 0.553, and it halved the number of chunks (294 before, 135 after: see the table in the next cell).

The overlap is 0.0: not one retrieved chunk is identical in both runs. That is expected. The chunks are fixed windows of 800 characters, so removing text shifts every later window. Even the same passage is cut differently. Reading the two best chunks shows that both runs found the same place in the paper, the list of ReAct's features on page 4.

The two answers say the same core thing: reasoning and acting happen inside one model, and thoughts are typed in language on top of the actions. So for this broad question, cleaning changed the chunks more than the answer.

### Extra (my own): which rule matters most?

The first reflection question asks which cleaning rule made the biggest difference. Instead of guessing, this cell applies **one rule at a time** to the raw text and measures retrieval again. It only chunks, embeds and retrieves (no model call), but it embeds the paper nine times, so it takes about ten minutes on a CPU. The last row is the plain `pypdf` text from the lesson notebook, as a reference.

In [11]:
from pypdf import PdfReader


def retrieval_only(text, query=QUERY, k=3):
    """The retrieval half of run_rag: no model call."""
    chunks = chunk_text(text)
    vectors = embedder.encode(chunks, normalize_embeddings=True)
    hits = retrieve(query, chunks, vectors, k=k)
    return len(chunks), mean([score for _, score in hits])


pypdf_text = "\n".join(p.extract_text() or "" for p in PdfReader(PDF).pages)

variants = [("raw (no rule)", raw_text)]
variants += [(rule.__name__ + " alone", rule(raw_text)) for rule in RULES]
variants += [("all six rules (clean_text)", clean_text)]
variants += [("pypdf text from the lesson", pypdf_text)]

ablation = []
print(f"{'text':<36}{'characters':>11}{'chunks':>8}{'mean top-3':>12}")
for name, text in variants:
    n_chunks, score = retrieval_only(text)
    ablation.append((name, len(text), n_chunks, score))
    print(f"{name:<36}{len(text):>11,}{n_chunks:>8}{score:>12.3f}")

text                                 characters  chunks  mean top-3


raw (no rule)                           205,117     294       0.510


drop_glyph_codes alone                  149,445     214       0.534


unwrap_tables alone                     153,624     220       0.542


drop_header_and_page_numbers alone      203,751     292       0.547


drop_short_lines alone                  204,771     293       0.519


join_hyphenated_words alone             205,069     293       0.510


collapse_blank_lines alone              205,117     294       0.510


all six rules (clean_text)               94,370     135       0.553


pypdf text from the lesson              110,116     158       0.584


**Reading the table.** The raw text scores 0.510. One rule at a time:

| Rule alone | Characters removed | Mean top-3 score |
|---|---:|---:|
| drop the header and page numbers | 1,366 | 0.547 |
| unwrap tables | 51,493 | 0.542 |
| drop glyph codes | 55,672 | 0.534 |
| drop short lines | 346 | 0.519 |
| join hyphenated words | 48 | 0.510 |
| collapse blank lines | 0 | 0.510 |

Dropping the header and the page numbers moved the score most (+0.037), although it removes only 1,366 characters. The two rules that remove the most text come next. Joining hyphenated words and collapsing blank lines changed nothing measurable here: this raw text has no runs of blank lines at all. All six rules together reach 0.553.

**A caution.** Every removal also shifts where the later 800-character windows are cut. So part of each difference is the luck of the cut, not the rule itself.

**The reference row.** The plain `pypdf` text from the lesson scores highest (0.584). It kept the spaces between the words. For this PDF the choice of converter matters more than my cleaning.

## Exercise 4 (stretch): Score the answers with an LLM judge

Similarity scores tell you about retrieval, not about the final answer. A common way to
score answer quality without a reference answer is to use the model as a judge.

Write a function that asks the model to rate, from 1 to 5, how well an answer is
supported by the context it was given (its faithfulness). Run it for both the raw and
the cleaned answer and compare.

Hint: build a prompt that includes the question, the context, and the answer, and ask
the model to reply with a single number. Reuse the `Groq` client from the provided cell.

In [12]:
def judge_faithfulness(question, context, answer):
    """Ask the model to rate 1 to 5 how well the answer is supported by the context."""
    joined = "\n---\n".join(context)
    prompt = (
        "You are a strict grader. Rate from 1 to 5 how well the answer is "
        "supported by the context.\n"
        "5 = every statement of the answer is in the context.\n"
        "1 = most of the answer is not in the context.\n\n"
        f"Question: {question}\n\nContext:\n{joined}\n\nAnswer: {answer}\n\n"
        "Reply with a single integer from 1 to 5 and nothing else."
    )
    # the provided chat model is the "client" of this notebook
    reply = response_text(model.invoke([{"role": "user", "content": prompt}]))
    # keep the reply only if it is exactly one digit from 1 to 5
    match = re.fullmatch(r"\s*([1-5])\s*", reply)
    return int(match.group(1)) if match else None

In [13]:
raw_faithfulness = judge_faithfulness(
    QUERY, raw_result["chunks"], raw_result["answer"]
)
clean_faithfulness = judge_faithfulness(
    QUERY, clean_result["chunks"], clean_result["answer"]
)
print("raw faithfulness:    ", raw_faithfulness)
print("cleaned faithfulness:", clean_faithfulness)

raw faithfulness:     4
cleaned faithfulness: 4


The judge gives both answers 4 out of 5. By this grade, cleaning did not change how well the answer is backed by its context.

One grade from a small local model is a weak signal. Read it together with the retrieval scores and the chunks above, not instead of them.

## Conclusion and Reflection

- Which cleaning rule made the biggest difference to the mean retrieval score or the
  overlap, and why?
- Did cleaning change the answer, or only the chunks behind it? What does that tell you
  about how sensitive this question is to noise?
- The mean score is an imperfect metric. What would you need in order to measure answer
  quality properly, and which tools (for example RAGAS) formalise this?

### My answers (our run, local model)

**Which cleaning rule made the biggest difference to the mean retrieval score or the overlap, and why?**

Measured one rule at a time (the extra cell in Exercise 3): dropping the repeated header and the page numbers raised the mean score most, from 0.510 to 0.547. Those lines sit in the middle of the running text on every page, so they interrupt chunks that are otherwise about one topic. Unwrapping tables (0.542) and dropping glyph codes (0.534) follow. Each of these two removes more than 50,000 characters of padding and unreadable figure text, which is why the number of chunks halves (294 to 135). All six rules together reach 0.553. The overlap between raw and cleaned retrieval is 0.0, but with fixed 800-character windows any removal shifts all later chunks, so an overlap of exact chunks says little here.

**Did cleaning change the answer, or only the chunks behind it? What does that tell you about how sensitive this question is to noise?**

Mostly the chunks. Both answers describe ReAct the same way, and the judge gives both a faithfulness of 4. The question is broad ("what is ReAct"), and many passages of the paper can answer it. So it is not very sensitive to noise. A narrow question whose answer sits in one sentence would be far more sensitive: if that one sentence is cut or buried in junk, the answer is gone.

**The mean score is an imperfect metric. What would you need in order to measure answer quality properly, and which tools (for example RAGAS) formalise this?**

A small test set: questions, a reference answer for each, and the passages that contain it. With it, retrieval can be scored with hit rate, precision and recall, and Mean Reciprocal Rank, and the answers can be compared with the reference. Without it, a model as judge can rate faithfulness and answer relevance, as in Exercise 4. RAGAS packages these scores. Two more things are missing here: more than one question (my comparison rests on a single one), and a stronger or second judge, because a small local model grades roughly.